# Complete Anime Character Organization Pipeline (Master Notebook)

## Unified End-to-End Orchestration
This master notebook provides a unified, interactive execution interface for the complete end-to-end anime character organization pipeline:

```text
Input Image Directory
        │
        ▼
Stage 1: Dataset Audit & Duplicate Detection
        │
        ▼
Stage 2: Anime Face/Body Detection & Crop Preparation
        │
        ▼
Stage 3: CCIP Embedding Extraction & Normalization
        │
        ▼
Stage 4: Optimized HDBSCAN Clustering & Diagnostics
        │
        ▼
Stage 5: Non-Destructive Folder Materialization
        │
        ▼
Stage 6: Semantic Character Tag Prediction & Renaming (Optional)
```

### Centralized Execution Principles:
- **Centralized Parameter Cell**: All parameters, paths, thresholds, and operational flags are configured in the single cell below.
- **Portability**: Automatically falls back to environment variables (`ANIME_PIPELINE_INPUT_DIR`) or local project directories.
- **Optimized Defaults**: Clustering hyperparameters (`min_cluster_size=2`, `min_samples=1`, `cluster_selection_epsilon=0.50`, soft noise reassignment) maximize character grouping while preserving purity.
- **Non-Destructive**: Original source files are strictly untouched.

## 1. Centralized Pipeline Parameters & Configuration
Modify these parameters to configure the entire pipeline execution.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import Image as IPImage, display

# ==============================================================================
# PIPELINE CONFIGURATION & PARAMETERS
# ==============================================================================

# --- 1. Paths & Directories ---
# Source directory containing raw images to organize
# Override via environment variable ANIME_PIPELINE_INPUT_DIR if desired
INPUT_DIR = Path(os.getenv("ANIME_PIPELINE_INPUT_DIR", "./input_images")).expanduser().resolve()

# Base pipeline workspace directory where runs and reports are stored
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()

# --- 2. Stage 1: Dataset Audit & Duplicate Detection ---
# Maximum Hamming distance on 64-bit pHash for near-duplicate candidate detection
PHASH_THRESHOLD = 6

# --- 3. Stage 2: Face/Body Detection & Crop Preparation ---
# Keep False to preserve diverse poses/angles of characters for clustering
USE_PERCEPTUAL_REPRESENTATIVES = False

# Detection confidence thresholds and bounding box padding
HEAD_CONF_THRESHOLD = 0.35
PERSON_CONF_THRESHOLD = 0.30
HEAD_PADDING_RATIO = 0.25   # Generous padding around hair/accessories for CCIP
PERSON_PADDING_RATIO = 0.10
CROP_OUTPUT_SIZE = (512, 512)
CROP_FORMAT = "JPEG"

# --- 4. Stage 3: CCIP Character Similarity Embeddings ---
CCIP_MODEL = "ccip-caformer-24-randaug-pruned"
CCIP_IMAGE_SIZE = 384
BATCH_SIZE = 16
NORMALIZE_EMBEDDINGS = True

# --- 5. Stage 4: Optimized HDBSCAN Clustering ---
MIN_CLUSTER_SIZE = 2              # Allows pairs and triplets of characters to form clusters
MIN_SAMPLES = 1                   # Eliminates harsh reachability distance penalty on small clusters
CLUSTER_SELECTION_EPSILON = 0.50    # Merges points within visual character similarity threshold
CLUSTER_SELECTION_METHOD = "eom"

# Soft noise reassignment: recovers borderline noise points to nearest cluster centroid
REASSIGN_NOISE = True
MAX_REASSIGN_DISTANCE = 0.55

# Keep character images unified in one folder instead of splitting into _review folders
SEPARATE_REVIEW_FOLDERS = False

# --- 6. Stage 5: Non-Destructive Materialization ---
# Materialization mode: 'hardlink' (instant, 0 extra disk), 'copy', or 'symlink'
MATERIALIZATION_MODE = "hardlink"
ALLOW_COPY_FALLBACK = True
ON_EXISTING = "error"

# --- 7. Stage 6: Semantic Character Naming (Anime Tagger) ---
RUN_TAGGING = True                # Set to True to infer character names using PixAI/WD14
PRIMARY_TAGGER = "pixai"
MIN_CHARACTER_SCORE = 0.65        # Minimum model confidence score
MIN_NAME_SHARE = 0.30             # Minimum fraction of cluster images agreeing
MIN_TOP_MARGIN = 0.05             # Score margin over second candidate
CREATE_NAMED_OUTPUT = True        # Materialize an additional named folder tree

print("=" * 80)
print("ANIME CHARACTER ORGANIZATION PIPELINE CONFIGURATION LOADED")
print(f"  Input Directory:         {INPUT_DIR}")
print(f"  Project Directory:       {PROJECT_DIR}")
print(f"  HDBSCAN Parameters:      min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}, epsilon={CLUSTER_SELECTION_EPSILON}")
print(f"  Noise Reassignment:      enabled={REASSIGN_NOISE}, max_distance={MAX_REASSIGN_DISTANCE}")
print(f"  Materialization Mode:    {MATERIALIZATION_MODE} (copy_fallback={ALLOW_COPY_FALLBACK})")
print(f"  Semantic Tagging:        enabled={RUN_TAGGING} (primary={PRIMARY_TAGGER})")
print("=" * 80)

## Stage 1: Dataset Audit, Integrity Validation, and Duplicate Detection
Scans the input directory, verifies image readability, and builds SHA-256 and pHash duplicate indices.

In [ ]:
from anime_character_organizer import run_dataset_audit

print(">>> Running Stage 1: Dataset Audit...")
audit_res = run_dataset_audit(
    input_dir=INPUT_DIR if INPUT_DIR.exists() else Path(".").resolve(),
    project_dir=PROJECT_DIR,
    phash_threshold=PHASH_THRESHOLD,
    show_progress=True,
)

audit_summary = audit_res["summary"]
print()
print("Stage 1 Audit Summary:")
print(f"  Candidate Files:      {audit_summary['total_candidate_files']}")
print(f"  Valid Images:         {audit_summary['valid_images']}")
print(f"  Corrupt/Invalid:      {audit_summary['invalid_images']}")
print(f"  Exact Dup Groups:     {audit_summary['exact_duplicate_groups']}")
print(f"  Perceptual Dup Groups:{audit_summary['perceptual_duplicate_groups']}")

## Stage 2: Anime Face/Body Detection and Crop Preparation
Detects character heads and persons, expands bounding boxes with hair padding, and extracts identity crops.

In [ ]:
from anime_character_organizer import run_crop_preparation

print(">>> Running Stage 2: Crop Preparation...")
crop_res = run_crop_preparation(
    project_dir=PROJECT_DIR,
    previous_run_dir=audit_res["run_dir"],
    use_perceptual_representatives=USE_PERCEPTUAL_REPRESENTATIVES,
    head_conf_threshold=HEAD_CONF_THRESHOLD,
    person_conf_threshold=PERSON_CONF_THRESHOLD,
    head_padding_ratio=HEAD_PADDING_RATIO,
    person_padding_ratio=PERSON_PADDING_RATIO,
    crop_output_size=CROP_OUTPUT_SIZE,
    crop_format=CROP_FORMAT,
    show_progress=True,
)

crop_summary = crop_res["summary"]
print()
print("Stage 2 Crop Summary:")
print(f"  Selected for Detection:{crop_summary['selected_for_detection']}")
print(f"  Successful Crops:      {crop_summary['successful_crops']}")
print(f"  Review Required:       {crop_summary['review_required']}")

## Stage 3: CCIP Character Similarity Feature Extraction
Extracts metric learning embeddings using the CCIP CAFormer model and performs $L_2$ row normalization.

In [ ]:
from anime_character_organizer import run_embedding_extraction

print(">>> Running Stage 3: CCIP Feature Extraction...")
embed_res = run_embedding_extraction(
    project_dir=PROJECT_DIR,
    previous_run_dir=crop_res["run_dir"],
    ccip_model=CCIP_MODEL,
    ccip_image_size=CCIP_IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    show_progress=True,
)

embed_summary = embed_res["summary"]
print()
print("Stage 3 Embedding Summary:")
print(f"  Successful Embeddings: {embed_summary['successful_embeddings']}")
print(f"  Embedding Dimension:   {embed_summary['embedding_dim']}")

## Stage 4: HDBSCAN Clustering with Researched Parameters & Diagnostics
Clusters normalized embeddings using optimized HDBSCAN parameters, computes centroids, and reassigns soft noise.

In [ ]:
from anime_character_organizer import run_clustering
from anime_character_organizer.visualization.plots import (
    plot_cluster_size_distribution,
    plot_probability_and_outlier_distributions,
)

print(">>> Running Stage 4: HDBSCAN Clustering & Diagnostics...")
cluster_res = run_clustering(
    project_dir=PROJECT_DIR,
    previous_run_dir=embed_res["run_dir"],
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    reassign_noise=REASSIGN_NOISE,
    max_reassign_distance=MAX_REASSIGN_DISTANCE,
    separate_review_folders=SEPARATE_REVIEW_FOLDERS,
    low_probability_threshold=0.35,
    high_outlier_quantile=0.95,
    show_progress=True,
)

cluster_summary = cluster_res["summary"]
print()
print("Stage 4 Clustering Summary:")
print(f"  Estimated Clusters:    {cluster_summary['estimated_clusters']}")
print(f"  Clustered Images:      {cluster_summary['clustered_points']}")
print(f"  Remaining Noise Items: {cluster_summary['noise_points']}")
print(f"  Reassigned from Noise: {cluster_summary['reassigned_points']}")

plot_cluster_size_distribution(cluster_res["cluster_summary_df"])
plt.show()

## Stage 5: Non-Destructive Folder Materialization
Materializes organized character destination folders non-destructively using hard links or file copies.

In [ ]:
from anime_character_organizer import run_folder_materialization

print(">>> Running Stage 5: Non-Destructive Folder Materialization...")
materialize_res = run_folder_materialization(
    project_dir=PROJECT_DIR,
    previous_run_dir=cluster_res["run_dir"],
    materialization_mode=MATERIALIZATION_MODE,
    allow_hardlink_fallback_to_copy=ALLOW_COPY_FALLBACK,
    on_existing=ON_EXISTING,
    show_progress=True,
)

mat_summary = materialize_res["summary"]
print()
print("Stage 5 Materialization Summary:")
print(f"  Organized Output Dir:  {mat_summary['final_output_dir']}")
print(f"  Total Files Linked:    {mat_summary['total_files']}")
print(f"  Total Folders:         {mat_summary['total_folders']}")
print(f"  Successful Operations: {mat_summary['successful_operations']}")

## Stage 6: Semantic Cluster Naming with Anime Taggers (Optional)
Queries PixAI / WD14 anime taggers across representative cluster crops and generates an accepted character name rename plan.

In [ ]:
if RUN_TAGGING:
    from anime_character_organizer import run_cluster_naming

    print(">>> Running Stage 6: Semantic Character Naming with Anime Tagger...")
    naming_res = run_cluster_naming(
        project_dir=PROJECT_DIR,
        previous_clustering_run_dir=cluster_res["run_dir"],
        previous_materialization_run_dir=materialize_res["run_dir"],
        primary_tagger=PRIMARY_TAGGER,
        min_character_score=MIN_CHARACTER_SCORE,
        min_name_share=MIN_NAME_SHARE,
        min_top_margin=MIN_TOP_MARGIN,
        create_named_output=CREATE_NAMED_OUTPUT,
        show_progress=True,
    )

    naming_summary = naming_res["summary"]
    print()
    print("Stage 6 Semantic Naming Summary:")
    print(f"  Evaluated Clusters:    {naming_summary['total_clusters_evaluated']}")
    print(f"  Accepted Names:        {naming_summary['accepted_character_names']}")
    if naming_res["named_output_dir"]:
        print(f"  Named Output Tree:     {naming_res['named_output_dir']}")
else:
    print("Stage 6: Semantic tagging disabled by configuration.")

## Pipeline Execution Summary & Next Steps
All six stages of the pipeline have completed. The organized character images are accessible under `organized_output/`.

In [ ]:
print("=" * 80)
print("ANIME CHARACTER ORGANIZATION PIPELINE COMPLETED")
print("=" * 80)
print(f"Input Directory:              {INPUT_DIR}")
print(f"Audited Images:               {audit_summary.get('valid_images', 0)}")
print(f"Extracted Crops:              {crop_summary.get('successful_crops', 0)}")
print(f"Extracted Embeddings:         {embed_summary.get('successful_embeddings', 0)}")
print(f"Discovered Clusters:          {cluster_summary.get('estimated_clusters', 0)}")
print(f"Organized Destination:        {materialize_res.get('final_output_dir')}")
if RUN_TAGGING:
    print(f"Named Clusters Accepted:      {naming_summary.get('accepted_character_names', 0)}")
    if naming_res.get('named_output_dir'):
        print(f"Named Tree Destination:       {naming_res.get('named_output_dir')}")
print("=" * 80)